# L2: Building a Basic Agent UI

En esta lección, conectaremos un agente de LangChain a una interfaz de chat en React utilizando CopilotKit y el protocolo AG-UI. Este es el patrón estándar para conectar cualquier backend de agente a cualquier frontend.

## Objetivos de aprendizaje
1. **Ejecutar un agente de LangChain:** Iniciar un backend compatible con AG-UI usando FastAPI.
2. **Configurar CopilotKit:** Conectar un frontend de React a tu agente mediante `CopilotRuntime`.
3. **Intercambiar backends de agentes:** Alternar entre LangChain/OpenAI y Google ADK/Gemini sin modificar el código de la UI.

---

## Antes de comenzar
Esta lección utiliza el comando mágico `%%writefile` para guardar el código directamente en el proyecto del frontend en el disco. La aplicación en ejecución detectará esos cambios automáticamente, permitiéndote ver las actualizaciones de la interfaz en tiempo real a medida que avanzas por las celdas.

## Instalación de dependencias

En este entorno de aprendizaje, todas las dependencias del backend y frontend están preinstaladas. Ejecuta las siguientes celdas para confirmar que todo está listo.

> **¿Ejecutando fuera de este entorno?** Instala las dependencias del backend con `pip install -r requirements.txt`, y luego ejecuta `npm install` en el directorio `frontend/`.

In [ ]:
# Suprimir mensajes de advertencia
import warnings
warnings.filterwarnings("ignore")


In [ ]:
from helper import install_frontend
install_frontend()

## Construcción del endpoint del Agente con FastAPI

En esta sección, configuraremos el backend utilizando FastAPI para exponer nuestro agente de LangGraph bajo el protocolo AG-UI. Esto permitirá que la interfaz de chat se conecte y se comunique con el agente.

In [ ]:
from fastapi import FastAPI

# Dependencias de CopilotKit y AG-UI para el servidor AG-UI
from ag_ui_langgraph import add_langgraph_fastapi_endpoint
from copilotkit import LangGraphAGUIAgent
from langchain.agents import create_agent

# Helper simple que inicia el servidor y gestiona conflictos de puertos
from helper import start_server

# Construir el endpoint AG-UI en una aplicación FastAPI
app = FastAPI()
graph = create_agent("openai:gpt-4.1")
agent = LangGraphAGUIAgent(
    name="lesson2_agent",
    description="Lesson 2 chart agent",
    graph=graph,
)
add_langgraph_fastapi_endpoint(app=app, agent=agent, path="/")

# Iniciar el servidor
start_server(app, port=8002)

## Introducción a CopilotKit

`CopilotKitMiddleware()` es el componente que conecta tu agente a CopilotKit; permite que el modelo descubra y ejecute herramientas del frontend (que construirás en la L3). Sin este middleware, el agente solo puede visualizar las herramientas definidas en el backend.

### Componentes clave de CopilotKit
En esta lección utilizaremos tres piezas fundamentales:

* **CopilotRuntime:** Un puente seguro que conecta tu frontend con cualquier backend de agente.
* **CopilotKit:** Un proveedor de React que configura la conexión del runtime para tu aplicación.
* **CopilotChat:** Una interfaz de chat personalizable y lista para usar ("baterías incluidas") para tus agentes.

---

## Iniciar el Frontend
Utilizaremos una aplicación preconstruida como base para las lecciones. A medida que realices cambios, la aplicación los detectará automáticamente y los mostrará en tiempo real.


A continuación, ejecutaremos las celdas necesarias para iniciar el servidor de desarrollo y abrir la vista previa en vivo.

In [ ]:
from helper import start_frontend
start_frontend(port=3002)

In [ ]:
from helper import load_api_keys
load_api_keys()

## Definición del agente

En este paso, crearemos un agente de LangChain equipado con un modelo de OpenAI, un checkpoint de memoria y el middleware de CopilotKit.

> **Nota:** Puedes volver a ejecutar esta celda siempre que desees cambiar la configuración del agente. La asignación `agent.graph = ...` permite realizar una recarga en caliente (*hot-reload*) del grafo sin necesidad de reiniciar el servidor.

In [ ]:
from copilotkit import CopilotKitMiddleware

# Importaciones del agente de LangChain
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver

# Crear un agente de LangChain preconstruido
graph = create_agent(
    model=ChatOpenAI(model="gpt-4.1"),
    tools=[],
    middleware=[CopilotKitMiddleware()],
    checkpointer=MemorySaver(),
    system_prompt=("You are a helpful assistant"),
)

# Actualizar el grafo del agente (recarga en caliente)
agent.graph = graph
print("✓ Agent graph updated!")

Finalmente, puedes abrir una vista previa en vivo de la aplicación en ejecución.

In [ ]:
from helper import display_app
display_app(port=3002)

In [ ]:
%%writefile frontend/server.ts

import { serve } from "@hono/node-server";
import { LangGraphHttpAgent } from "@ag-ui/langgraph";
import {
  CopilotRuntime,
  createCopilotEndpoint,
} from "@copilotkit/runtime/v2";

// Configuración del agente que apunta al endpoint de FastAPI (puerto 8002)
const langGraphAgent = new LangGraphHttpAgent({
  url: process.env.LANGGRAPH_DEPLOYMENT_URL || "http://localhost:8002",
});

// Inicialización del runtime con el agente registrado como "default"
const runtime = new CopilotRuntime({
  agents: {
    default: langGraphAgent,
  },
});

// Creación del endpoint de API para CopilotKit
const app = createCopilotEndpoint({
  runtime,
  basePath: "/api/copilotkit",
});

// Inicio del servidor en el puerto 4002
serve({ fetch: app.fetch, port: 4002 }, () => {
  console.log("CopilotKit API server running at http://localhost:4002");
});

## Envolver la aplicación en el proveedor `CopilotKit`

El proveedor `CopilotKit` conecta tu aplicación React con el *runtime* a través de la propiedad `runtimeUrl`. Puedes colocarlo en `main.tsx` para envolver toda la aplicación, o alrededor de componentes individuales si lo prefieres.

In [ ]:
%%writefile frontend/src/main.tsx

import { StrictMode } from "react";
import { createRoot } from "react-dom/client";
import { CopilotKit } from "@copilotkit/react-core/v2";
import "@copilotkit/react-core/v2/styles.css";
import "./globals.css";
import App from "./App";

createRoot(document.getElementById("root")!).render(
  <StrictMode>
    <main className="h-screen w-screen">
      {/* El proveedor CopilotKit conecta el frontend al endpoint del runtime */}
      <CopilotKit runtimeUrl="/api/copilotkit" useSingleEndpoint={false}>
        <App />
      </CopilotKit>
    </main>
  </StrictMode>
);

## Configurar el componente `CopilotChat`

Finalmente, añadiremos el componente `CopilotChat` y lo apuntaremos al agente `default` que registramos anteriormente en el `CopilotRuntime`.

In [ ]:
%%writefile frontend/src/App.tsx

import { CopilotChat } from "@copilotkit/react-core/v2";

const agentId = "default";

export default function App() {
  return <CopilotChat agentId={agentId} />;
}

## ¡Pruébalo!

Tu aplicación ya cuenta con una interfaz de chat funcional. Ejecuta la celda a continuación para abrir la vista previa e interactuar con tu agente.

In [ ]:
from helper import display_app
display_app(port=3002)

Nota: También puedes usar CopilotKit en modo headless (sin una interfaz de chat preconstruida) a través del hook useAgent. En esta lección, utilizaremos el componente incorporado CopilotChat por simplicidad.